In [1]:
import pandas as pd
from pathlib import Path
from tqdm import tqdm
import numpy as np
from sklearn.metrics import precision_recall_fscore_support
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.patches as mpatches
from joblib import Parallel, delayed
from scipy.stats import hmean
from matplotlib.ticker import PercentFormatter
import re

In [3]:
nifd_path = Path('/projectnb/vkolagrp/bellitti/adrd-foundation-model/adrd_simplified_evaluation/results/NIFD/test_cog')
nacc_path = Path('/projectnb/vkolagrp/bellitti/adrd-foundation-model/adrd_simplified_evaluation/results/nacc_test_updated/test_cog')
adni_path = Path('/projectnb/vkolagrp/bellitti/adrd-foundation-model/adrd_simplified_evaluation/results/ADNI/test_cog')
ppmi_path = Path('/projectnb/vkolagrp/bellitti/adrd-foundation-model/adrd_simplified_evaluation/results/PPMI/test_cog')
brainlat_path = Path('/projectnb/vkolagrp/bellitti/adrd-foundation-model/adrd_simplified_evaluation/results/brainlat/test_cog')

In [4]:
def option_string_to_dict(options):
    pattern = r"([A-Z])\. ([^\n]+)"
    matches = re.findall(pattern, options)
    return {key: value for key, value in matches}

def load_answers(dir_path, dataset_name):
    # load all parquet files from the directory, stack them into a pandas datafame
    # this only reads the participant ID, ground trush answer and the prediction. 
    # Reading only those columns is significantly faster (about 100x) than loading the whole dataframe.
    # Loading everything is very slow because there are extremely long strings (model outputs) in some columns

    fpaths = list(dir_path.rglob('*.parquet'))

    dfs = []

    cols_to_read = ['ID','ground_truth','prediction','ground_truth_text','options']

    for fpath in tqdm(fpaths):

        model = fpath.parent.name.split('-',3)[-1] 
        benchmark = fpath.parent.parent.name.split('_',1)[-1].upper()

        df = pd.read_parquet(fpath,columns=cols_to_read)
    
        df = df.assign(model=model, benchmark=benchmark)

        df['correct'] = (df['ground_truth'] == df['prediction']).astype(int)

        df['prediction_text'] = df.apply(lambda row: option_string_to_dict(row['options']).get(row['prediction'],'invalid'),axis=1)

        dfs.append(df)

    df = pd.concat(dfs)
    df['dataset'] = dataset_name

    # make these columns Categorical
    group_cols = ["dataset", "benchmark", "model", "ground_truth_text", 'prediction_text']
    for col in group_cols:
        df[col] = pd.Categorical(df[col])

    return df


In [5]:
nacc_res = load_answers(nacc_path,dataset_name='NACC')

  0%|          | 0/9 [00:00<?, ?it/s]

100%|██████████| 9/9 [00:11<00:00,  1.28s/it]


In [6]:
blat_res = load_answers(brainlat_path,dataset_name='BrainLat')

  0%|          | 0/8 [00:00<?, ?it/s]

100%|██████████| 8/8 [00:00<00:00, 12.31it/s]


In [7]:
ppmi_res = load_answers(ppmi_path,dataset_name='PPMI')

100%|██████████| 8/8 [00:01<00:00,  4.66it/s]


In [8]:
nifd_res = load_answers(nifd_path,dataset_name='NIFD')

  0%|          | 0/8 [00:00<?, ?it/s]

100%|██████████| 8/8 [00:00<00:00, 15.85it/s]


In [9]:
adni_res = load_answers(adni_path,dataset_name='ADNI')

100%|██████████| 8/8 [00:01<00:00,  6.31it/s]


In [10]:
nifd_res.sample()

,ID,ground_truth,prediction,ground_truth_text,options,model,benchmark,correct,prediction_text,dataset
1161,1_S_0154,C,C,Normal Cognition (NC),A. Mild Cognitive Impairment (MCI)\nB. Dementi...,Qwen2.5-7B-Instruct,COG,1,Normal Cognition (NC),NIFD


In [11]:
def _process_group(id, group, n_boot, seed):
    """
    Helper function for parallel processing.
    Computes bootstrap CIs for precision and recall by class.
    """
    dataset, model = id
    n_samples = len(group)
    
    if n_samples == 0:
        return []

    unique_labels = group['ground_truth_text'].unique()
    
    # --- Reproducible RNG per worker ---
    rng = np.random.default_rng(seed)
    
    # Pre-generate all bootstrap indices at once
    bootstrap_indices = rng.integers(0, n_samples, size=(n_boot, n_samples))
    
    # Pre-allocate dicts to store results per class
    # Keys will be (metric, class_label)
    boot_results = {}
    for label in unique_labels:
        boot_results[('precision', label)] = []
        boot_results[('recall', label)] = []
    
    # Convert to NumPy arrays ONCE per group
    y_true = group["ground_truth_text"].to_numpy()
    y_pred = group["prediction_text"].to_numpy()
    
    for indices in bootstrap_indices:
        y_true_boot = y_true[indices]
        y_pred_boot = y_pred[indices]

        # Get per-class metrics
        p, r, f, s = precision_recall_fscore_support(
            y_true_boot,
            y_pred_boot,
            average=None,
            labels=unique_labels,
            zero_division=0,
        )
        
        # Store per-class results
        for i, label in enumerate(unique_labels):
            boot_results[('precision', label)].append(p[i])
            boot_results[('recall', label)].append(r[i])

    # --- Calculate quantiles ---
    res_list = []
    for label in unique_labels:
        # Process precision for this class
        precision_values = np.array(boot_results[('precision', label)])
        
        low_idx = int(0.025 * n_boot)
        med_idx = int(0.5 * n_boot)
        high_idx = int(0.975 * n_boot)
        
        partitioned = np.partition(precision_values, [low_idx, med_idx, high_idx])
        
        res_list.append({
            "dataset": dataset,
            "model": model,
            "class": label,
            "metric": "precision",
            "median": partitioned[med_idx],
            "low": partitioned[low_idx],
            "high": partitioned[high_idx],
        })
        
        # Process recall for this class
        recall_values = np.array(boot_results[('recall', label)])
        partitioned = np.partition(recall_values, [low_idx, med_idx, high_idx])
        
        res_list.append({
            "dataset": dataset,
            "model": model,
            "class": label,
            "metric": "recall",
            "median": partitioned[med_idx],
            "low": partitioned[low_idx],
            "high": partitioned[high_idx],
        })
        
    return res_list


def optimized_bootstrap_parallel(df, n_boot, seed=None, n_jobs=-1):
    """
    Compute bootstrap confidence intervals for classwise precision and recall in parallel.
    
    Parameters
    ----------
    df : pd.DataFrame
        Input dataframe with columns: dataset, model, ground_truth_text, prediction_text
    n_boot : int
        Number of bootstrap samples
    seed : int, optional
        Random seed for reproducible results
    n_jobs : int, optional
        Number of CPU cores to use. -1 means all cores.
    
    Returns
    -------
    pd.DataFrame
        Results with columns: dataset, model, class, metric, median, low, high
    """
    
    # --- Setup ---
    main_rng = np.random.default_rng(seed)
    
    df_copy = df.copy()
    group_cols = ["dataset", "model"]
            
    # Get all groups
    groups = list(df_copy.groupby(group_cols, observed=True))
    
    # Generate a unique seed for each group
    n_groups = len(groups)
    group_seeds = main_rng.integers(0, 2**32, size=n_groups)
    
    # --- Parallel Execution ---
    results_list_of_lists = Parallel(n_jobs=n_jobs, verbose=0)(
        delayed(_process_group)(
            id, 
            group, 
            n_boot, 
            group_seeds[i]
        )
        for i, (id, group) in enumerate(groups)
    )
    
    # --- Collect Results ---
    final_results = [item for sublist in results_list_of_lists for item in sublist]
    
    return pd.DataFrame(final_results)

In [12]:
nifd_metrics = optimized_bootstrap_parallel(
    df=nifd_res,
    n_boot=100,
    seed=42,
    n_jobs=32
)

In [13]:
blat_metrics = optimized_bootstrap_parallel(
    df=blat_res,
    n_boot=100,
    seed=42,
    n_jobs=32
)

In [14]:
adni_metrics = optimized_bootstrap_parallel(
    df=adni_res,
    n_boot=100,
    seed=42,
    n_jobs=32
)

In [15]:
nacc_metrics = optimized_bootstrap_parallel(
    df=nacc_res,
    n_boot=10,
    seed=42,
    n_jobs=32
)

In [16]:
ppmi_metrics = optimized_bootstrap_parallel(
    df=ppmi_res,
    n_boot=100,
    seed=42,
    n_jobs=32
)

In [17]:
all_metrics = pd.concat([adni_metrics,nifd_metrics,nacc_metrics,ppmi_metrics, blat_metrics])

In [18]:
all_metrics.sample(10)

,dataset,model,class,metric,median,low,high
9,NACC,NACC-3B-OS,Normal Cognition (NC),recall,0.814836,0.812474,0.816412
23,NACC,NACC-3B-SCE,Dementia (DE),recall,0.826504,0.824592,0.828739
4,BrainLat,NACC-3B-OS,Dementia (DE),precision,0.931366,0.919251,0.940881
8,NACC,NACC-3B-OS,Normal Cognition (NC),precision,0.926842,0.925329,0.928599
14,PPMI,NACC-3B-OS-SCE,Normal Cognition (NC),precision,0.942710,0.937816,0.946486
14,ADNI,NACC-3B-OS-SCE,Mild Cognitive Impairment (MCI),precision,0.591455,0.569562,0.611851
3,NIFD,NACC-3B,Normal Cognition (NC),recall,0.699844,0.665158,0.740580
36,NACC,Qwen2.5-3B-Instruct,Mild Cognitive Impairment (MCI),precision,0.285270,0.282245,0.288597
18,ADNI,NACC-3B-SCE,Normal Cognition (NC),precision,0.663193,0.643899,0.677019
37,ADNI,Qwen2.5-3B-Instruct,Normal Cognition (NC),recall,0.637487,0.620941,0.653266


In [ ]:
import os
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

sns.set_style("whitegrid")

# Use 'all_metrics' dataframe which contains results from multiple datasets
# It should have columns: dataset, model, class, metric, median, low, high

# Map model names to abbreviations
model_map = {
    "Qwen2.5-3B-Instruct": "Q3B",
    "NACC-3B-OS-SCE": "LUNAR",
    "Qwen2.5-7B-Instruct": "Q7B",
}
all_metrics["model_abbrev"] = all_metrics["model"].map(model_map)

# Map class names to abbreviations
class_map = {
    'Mild Cognitive Impairment (MCI)': 'MCI',
    'Cognitively Impaired (IMP)': 'IMP',
    'Dementia (DE)': 'DE',
    'Normal Cognition (NC)': 'NC'
}
all_metrics["class_abbrev"] = all_metrics["class"].map(class_map).fillna(all_metrics["class"])

# Filter for precision and recall only
df = all_metrics[all_metrics['metric'].isin(['precision', 'recall'])].copy()

# Get unique values
datasets = sorted(df["dataset"].unique())
models = ["Q3B", "LUNAR", "Q7B"]

# Define colors for models
palette = dict(zip(models, sns.color_palette("colorblind", n_colors=len(models))))

# Create figure: rows = metrics, columns = datasets
metrics = ['precision', 'recall']
metric_names = {'precision': 'Precision', 'recall': 'Recall'}
n_datasets = len(datasets)

fig, axes = plt.subplots(2, n_datasets, figsize=(4 * n_datasets, 6))

# Ensure axes is 2D even with one dataset
if n_datasets == 1:
    axes = axes.reshape(-1, 1)

# --- New fixed orders ---
class_order = ["NC", "MCI", "IMP", "DE"]
dataset_order = ["NACC", "ADNI", "BrainLat", "NIFD", "PPMI"]

# Override datasets list
datasets = [d for d in dataset_order if d in df["dataset"].unique()]

for col_idx, dataset in enumerate(datasets):
    # Get classes for this specific dataset (using abbreviated names)
    present = df[df['dataset'] == dataset]['class_abbrev'].unique()
    dataset_classes = [c for c in class_order if c in present]
    
    for row_idx, metric in enumerate(metrics):
        ax = axes[row_idx, col_idx]
        
        # Filter data for this dataset and metric
        df_subset = df[(df['dataset'] == dataset) & (df['metric'] == metric)]
        
        # Set up bar positions
        n_models = len(models)
        n_classes = len(dataset_classes)
        bar_width = 0.8 / n_models
        
        # For each model, plot bars
        for i, model in enumerate(models):
            df_model = df_subset[df_subset['model_abbrev'] == model]
            
            # Get values in class order
            values = []
            errors_low = []
            errors_high = []
            
            for cls in dataset_classes:
                cls_data = df_model[df_model['class_abbrev'] == cls]
                if len(cls_data) > 0:
                    values.append(cls_data['median'].values[0])
                    errors_low.append(cls_data['median'].values[0] - cls_data['low'].values[0])
                    errors_high.append(cls_data['high'].values[0] - cls_data['median'].values[0])
                else:
                    values.append(0)
                    errors_low.append(0)
                    errors_high.append(0)
            
            # Calculate bar positions
            x_pos = np.arange(n_classes) + i * bar_width - (n_models - 1) * bar_width / 2
            
            # Plot bars
            bars = ax.bar(
                x_pos,
                values,
                bar_width,
                label=model if row_idx == 0 and col_idx == n_datasets - 1 else "",
                color=palette[model],
                alpha=0.8,
                yerr=[errors_low, errors_high],
                capsize=3,
                error_kw={'linewidth': 1.5}
            )
            
            # Annotate bars with values
            for bar, val in zip(bars, values):
                if val > 0:  # Only annotate non-zero bars
                    height = bar.get_height()
                    ax.text(
                        bar.get_x() + bar.get_width() / 2,
                        height + 0.07,
                        f"{val:.2f}",
                        ha="center",
                        va="bottom",
                        fontsize=7,
                        # color=palette[model]
                        color="black",
                    )
        
        # Formatting
        # ax.set_xlabel("Class", fontsize=10)
        ax.set_ylabel(metric_names[metric], fontsize=10)
        
        # Title: metric name on left column, dataset name on top row
        if row_idx == 0:
            ax.set_title(f"{dataset}", fontsize=11, fontweight='bold')
        # if col_idx == 0:
        #     ax.text(-0.3, 0.5, metric_names[metric], transform=ax.transAxes,
        #            fontsize=11, fontweight='bold', va='center', rotation=90)
        
        # ax.set_xticks(np.arange(n_classes))
        # ax.set_xticklabels(dataset_classes, rotation=45, ha='right')

        # Use fixed-order class labels
        ax.set_xticks(np.arange(len(dataset_classes)))
        ax.set_xticklabels(dataset_classes, rotation=45, ha='right')

        ax.set_ylim(0, 1.2)  # Extra space for annotations
        ax.grid(True, alpha=0.3, axis='y')
        
        # Remove spines
        sns.despine(ax=ax, left=True, bottom=True, right=True, top=True)
        # Re-add spines only for NACC panels
        if dataset == "NACC":
            for spine in ["left", "bottom", "right", "top"]:
                ax.spines[spine].set_visible(True)
                ax.spines[spine].set_linewidth(1.0)


# Add legend (horizontal, 3 columns, below the plots)
handles = [
    plt.Rectangle((0, 0), 1, 1, fc=palette[m], alpha=0.8, label=m)
    for m in models
]
fig.legend(handles=handles, title="Model", loc="lower center", ncol=3, 
           bbox_to_anchor=(0.15, -0.053), frameon=True)

plt.tight_layout(rect=[0, 0.02, 1, 1])

# Save figure
os.makedirs("../figure2_classwise", exist_ok=True)
filename = "../figure2_classwise/classwise_precision_recall.pdf"
plt.savefig(filename, bbox_inches="tight", dpi=300)
print(f"Saved {filename}")
plt.close()

Saved ../figure2_classwise/classwise_precision_recall.pdf
